In [53]:
import os
import re
import glob
import json
import argparse
import warnings
from pathlib import Path
import spikeinterface.full as si
import spikeinterface.sorters as ss

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [54]:
### DATASET-SPECIFIC OUTPUT NAMING
# ============================================================

def make_dataset_label_from_raw_path(raw_h5_file, anchor_folder="Data"):
    """
    Create a readable dataset label from the raw H5 path.

    Example:
        ../Data/FT_SN_FabryLinesH9/260527/T003708/Network/000039/data.raw.h5

    becomes:
        FT_SN_FabryLinesH9_260527_T003708_Network_000039

    If 'Data' is not found in the path, it falls back to the last folders
    before the file name.
    """
    p = Path(raw_h5_file)
    parts = list(p.parts)

    if anchor_folder in parts:
        anchor_index = parts.index(anchor_folder)
        label_parts = parts[anchor_index + 1:-1]
    else:
        label_parts = parts[-6:-1]

    if len(label_parts) == 0:
        label_parts = [p.name.replace(".raw.h5", "").replace(".h5", "")]

    label = "_".join(label_parts)

    label = re.sub(r"[^A-Za-z0-9._-]+", "_", label)
    label = re.sub(r"_+", "_", label).strip("_")

    if label == "":
        label = "unknown_dataset"

    return label


def make_dataset_output_paths(raw_h5_file, output_dir):
    """
    Build all dataset-specific output folders.

    Every saved folder name includes the dataset label.
    """
    dataset_label = make_dataset_label_from_raw_path(raw_h5_file)

    plot_dir = os.path.join(
        output_dir,
        f"plots_{dataset_label}"
    )

    os.makedirs(plot_dir, exist_ok=True)

    return dataset_label, plot_dir


In [55]:
### BASIC HELPERS
# ============================================================

def clean_inf(df):
    df = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    return df


def safe_filename(name):
    name = str(name)
    name = name.replace("/", "_").replace("\\", "_").replace(" ", "_")
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


def save_or_show(filename=None, show=False, dpi=200):
    plt.tight_layout()

    if filename is not None:
        plt.savefig(filename, dpi=dpi)

    if show:
        plt.show()

    plt.close()


def ensure_unit_id_column(df):
    df = df.copy()

    if "unit_id" not in df.columns:
        first_col = df.columns[0]
        df = df.rename(columns={first_col: "unit_id"})

    return df


def load_master(master_path, output_dir):
    """
    Load master_phenotype_matrix.csv if it exists.

    If it does not exist, build it from *_phenotype_features.csv files in output_dir.
    """
    if os.path.exists(master_path):
        master = pd.read_csv(master_path)
        master = ensure_unit_id_column(master)
        return clean_inf(master)

    phenotype_files = sorted(
        glob.glob(os.path.join(output_dir, "*_phenotype_features.csv"))
    )

    if len(phenotype_files) == 0:
        raise FileNotFoundError(
            "Could not find master_phenotype_matrix.csv or *_phenotype_features.csv files."
        )

    dfs = []

    for path in phenotype_files:
        df = pd.read_csv(path)
        df = ensure_unit_id_column(df)

        if "safe_id" not in df.columns:
            safe_id = os.path.basename(path).replace("_phenotype_features.csv", "")
            df["safe_id"] = safe_id

        dfs.append(df)

    master = pd.concat(dfs, ignore_index=True)
    master = clean_inf(master)
    master.to_csv(master_path, index=False)

    return master


def add_safe_id_to_well_summary(well_summary, master):
    """
    Ensure well_summary has a safe_id column.

    safe_id should look like:
        rec0000_well000

    If the existing well_level_summary.csv lacks safe_id, reconstruct it from
    recording_id + well_id, or merge it from the master phenotype table.
    """
    well_summary = well_summary.copy()

    if "safe_id" in well_summary.columns:
        return well_summary

    print("well_summary has no safe_id column. Reconstructing safe_id...")

    # Best case: recording_id and well_id are present.
    if "recording_id" in well_summary.columns and "well_id" in well_summary.columns:
        well_summary["safe_id"] = (
            well_summary["recording_id"].astype(str)
            + "_"
            + well_summary["well_id"].astype(str)
        )
        return well_summary

    # Fallback: merge from master using recording_id/well_id if available.
    merge_cols = []

    for col in ["recording_id", "well_id"]:
        if col in well_summary.columns and col in master.columns:
            merge_cols.append(col)

    if len(merge_cols) > 0 and "safe_id" in master.columns:
        safe_map = (
            master[merge_cols + ["safe_id"]]
            .drop_duplicates()
        )

        well_summary = well_summary.merge(
            safe_map,
            on=merge_cols,
            how="left"
        )

        if "safe_id" in well_summary.columns and well_summary["safe_id"].notna().any():
            return well_summary

    raise ValueError(
        "Could not reconstruct safe_id for well_summary. "
        "Need either safe_id column, or recording_id + well_id columns."
    )


def load_well_summary(master, well_summary_path):
    """
    Load well_level_summary.csv if it exists.

    If it does not exist, generate a minimal one from the master table.

    Always ensures safe_id is present.
    """
    if os.path.exists(well_summary_path):
        well_summary = pd.read_csv(well_summary_path)
        well_summary = clean_inf(well_summary)
        well_summary = add_safe_id_to_well_summary(well_summary, master)
        return well_summary

    group_cols = ["safe_id"]

    for col in ["recording_id", "well_id", "condition", "celltype"]:
        if col in master.columns and col not in group_cols:
            group_cols.append(col)

    agg = {
        "n_units": ("unit_id", "count"),
        "mean_snr": ("snr", "mean"),
        "median_snr": ("snr", "median"),
        "mean_firing_rate_hz": ("firing_rate", "mean"),
        "median_firing_rate_hz": ("firing_rate", "median"),
    }

    if "burst_rate_hz" in master.columns:
        agg["mean_burst_rate_hz"] = ("burst_rate_hz", "mean")
        agg["median_burst_rate_hz"] = ("burst_rate_hz", "median")

    if "mean_network_sttc" in master.columns:
        agg["mean_network_sttc"] = ("mean_network_sttc", "mean")
        agg["median_network_sttc"] = ("mean_network_sttc", "median")

    well_summary = (
        master.groupby(group_cols, dropna=False)
        .agg(**agg)
        .reset_index()
    )

    well_summary = add_safe_id_to_well_summary(well_summary, master)

    well_summary.to_csv(well_summary_path, index=False)

    return clean_inf(well_summary)


def pick_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def get_safe_ids(master):
    if "safe_id" not in master.columns:
        raise ValueError("master table must contain a 'safe_id' column.")

    return sorted(master["safe_id"].dropna().unique())


def load_sttc_matrix(safe_id, output_dir):
    path = os.path.join(output_dir, f"{safe_id}_sttc_matrix.csv")

    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing STTC matrix: {path}")

    sttc = pd.read_csv(path, index_col=0)

    sttc.index = sttc.index.astype(str)
    sttc.columns = sttc.columns.astype(str)

    return sttc

import re


def short_safe_id_label(x):
    """
    Return only the recXXXX_wellXXX part from long labels.

    Examples:
        rec0000_well000 -> rec0000_well000
        FT_SN_..._rec0000_well000 -> rec0000_well000
        well000 -> well000
    """
    x = str(x)

    m = re.search(r"(rec\d+_well\d+)", x)
    if m:
        return m.group(1)

    m = re.search(r"(well\d+)", x)
    if m:
        return m.group(1)

    return x

def _get_well_column(master):
    """
    Prefer well_id if present, otherwise fall back to safe_id.
    """
    if "well_id" in master.columns:
        return "well_id"
    if "safe_id" in master.columns:
        return "safe_id"

    return None


def _well_sort_key(well):
    """
    Natural-ish sorting for names like well000, well001, rec0000_well000, etc.
    """
    import re

    well = str(well)
    nums = re.findall(r"\d+", well)

    if len(nums) == 0:
        return (well,)

    return tuple(int(x) for x in nums)


import re
import matplotlib.patches as mpatches


def _well_sort_key(well):
    """
    Sorts well names naturally, e.g.
    well000, well001, ..., well023
    rec0000_well000, rec0000_well001, ...
    """
    well = str(well)
    nums = re.findall(r"\d+", well)

    if len(nums) == 0:
        return (well,)

    return tuple(int(x) for x in nums)


def _build_well_label_maps(df, well_col, well_label_config=None):
    """
    Builds:
        color_by_well = {well_id: color}
        label_by_well = {well_id: label}

    well_label_config format:
        {
            "LABEL": {
                "wells": [1, 2, 3],
                "color": "tab:blue",
            },
            ...
        }

    Integer wells are treated as 1-based positions after sorting wells.
    So if the dataframe has well000 ... well023:
        1  -> well000
        24 -> well023

    String wells are matched directly.
    """

    if well_label_config is None:
        return {}, {}

    wells_sorted = sorted(
        df[well_col].astype(str).unique(),
        key=_well_sort_key,
    )

    color_by_well = {}
    label_by_well = {}

    for label, spec in well_label_config.items():
        wells = spec.get("wells", [])
        color = spec.get("color", None)

        for well in wells:
            if isinstance(well, int):
                if well < 1 or well > len(wells_sorted):
                    warnings.warn(
                        f"Well index {well} for label '{label}' is outside "
                        f"the available 1-{len(wells_sorted)} range."
                    )
                    continue

                well_key = wells_sorted[well - 1]

            else:
                well_key = str(well)

            color_by_well[well_key] = color
            label_by_well[well_key] = label

    return color_by_well, label_by_well



In [56]:
### QC distribution PLOTS per unit (all) : SNR, FR, QC-passing units  
# ============================================================


def plot_snr_distribution(master, plot_dir, font_size = 12, show=False, bins=50):
    if "snr" not in master.columns:
        warnings.warn("Skipping SNR plot: column 'snr' not found.")
        return

    x = master["snr"].dropna()
    plt.rcParams.update({"font.size": font_size, "svg.fonttype": "none"})
    plt.figure()
    plt.hist(x, bins=bins)
    plt.xlabel("SNR")
    plt.ylabel("Number of units")
    plt.title("SNR distribution of QC-passing units", fontsize = font_size)

    save_or_show(
        filename=os.path.join(plot_dir, "unit_snr_distribution.svg"),
        show=show,
    )



def plot_firing_rate_distribution(master, plot_dir, show=False, font_size = 12, bins=50, title_extra = None):
    if "firing_rate" not in master.columns:
        warnings.warn("Skipping firing-rate plot: column 'firing_rate' not found.")
        return

    x = master["firing_rate"].dropna()
    plt.rcParams.update({"font.size": font_size, "svg.fonttype": "none"})

    plt.figure()
    plt.hist(x, bins=bins)
    plt.xlabel("Firing rate (Hz)")
    plt.ylabel("Number of units")
    if (title_extra):
        plt.title("Firing-rate distribution of QC-passing units" + f'{title_extra}')
    else: 
        plt.title("Firing-rate distribution of QC-passing units")

    save_or_show(
        filename=os.path.join(plot_dir, "unit_firing_rate_distribution.svg"),
        show=show,
    )



def plot_units_per_well(
    master,
    plot_dir,
    table_dir,
    show=False,
    font_size = 12,
    well_label_config=None,
):
    well_col = _get_well_column(master)

    if well_col is None:
        warnings.warn("Skipping units-per-well plot: neither 'well_id' nor 'safe_id' found.")
        return

    counts = (
        master.groupby(well_col)
        .size()
        .reset_index(name="n_units")
    )

    counts[well_col] = counts[well_col].astype(str)

    counts = counts.sort_values(
        by=well_col,
        key=lambda s: s.map(_well_sort_key)
    )

    counts.to_csv(
        os.path.join(table_dir, "units_per_well.csv"),
        index=False
    )

    color_by_well, label_by_well = _build_well_label_maps(
        df=counts,
        well_col=well_col,
        well_label_config=well_label_config,
    )

    bar_colors = [
        color_by_well.get(well, None)
        for well in counts[well_col]
    ]

    x_labels = [
        short_safe_id_label(well).replace("rec0000_", "")
        for well in counts[well_col]
    ]
    plt.rcParams.update({"font.size": font_size, "svg.fonttype": "none"})

    plt.figure(figsize=(max(10, len(counts) * 0.45), 5))

    if any(c is not None for c in bar_colors):
        plt.bar(x_labels, counts["n_units"], color=bar_colors)
    else:
        plt.bar(x_labels, counts["n_units"])

    plt.xlabel("Well")
    plt.ylabel("QC-passing units")
    plt.title("QC-passing units per well")
    plt.xticks(rotation=90)
    
    if well_label_config is not None:
        legend_handles = []

        for label, spec in well_label_config.items():
            color = spec.get("color", None)
            if color is not None:
                legend_handles.append(
                    mpatches.Patch(color=color, label=label)
                )

        if len(legend_handles) > 0:
            plt.legend(
                handles=legend_handles,
                title="Group",
                bbox_to_anchor=(1.02, 1),
                loc="upper left",
                borderaxespad=0,
            )
            plt.tight_layout(rect=[0, 0, 0.85, 1])
        else:
            plt.tight_layout()
    else:
        plt.tight_layout()

    save_or_show(
        filename=os.path.join(plot_dir, "well_qc_passing_units.svg"),
        show=show,
    )


In [57]:
### QC distribution PLOTS per unit (per well): SNR, FR, QC-passing units
# ============================================================


def _plot_distribution_per_well_grid(
    master,
    value_col,
    plot_dir,
    filename,
    xlabel,
    title,
    show=False,
    bins=50,
    expected_n_wells=24,
):
    well_col = _get_well_column(master)

    if well_col is None:
        warnings.warn(
            f"Skipping {value_col} per-well plot: neither 'well_id' nor 'safe_id' found."
        )
        return

    if value_col not in master.columns:
        warnings.warn(
            f"Skipping {value_col} per-well plot: column '{value_col}' not found."
        )
        return

    df = master.copy()
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df = df.dropna(subset=[value_col, well_col])

    if df.empty:
        warnings.warn(
            f"Skipping {value_col} per-well plot: no finite values found."
        )
        return

    wells_present = sorted(df[well_col].unique(), key=_well_sort_key)

    if len(wells_present) != expected_n_wells:
        warnings.warn(
            f"Expected {expected_n_wells} wells, but found {len(wells_present)} "
            f"unique values in '{well_col}'. Plotting available wells."
        )

    # Use available wells, but keep a 4 x 6 layout.
    wells = wells_present[:expected_n_wells]

    all_values = df[value_col].to_numpy(dtype=float)

    # Shared x-axis bin edges across all wells
    bin_edges = np.histogram_bin_edges(all_values, bins=bins)

    # Shared y-axis limit across all wells
    max_count = 0
    for well in wells:
        x = df.loc[df[well_col] == well, value_col].to_numpy(dtype=float)
        counts, _ = np.histogram(x, bins=bin_edges)
        if len(counts) > 0:
            max_count = max(max_count, counts.max())

    y_max = max_count * 1.10 if max_count > 0 else 1

    fig, axes = plt.subplots(
        nrows=4,
        ncols=6,
        figsize=(20, 12),
        sharex=True,
        sharey=True,
    )

    axes = axes.ravel()

    for ax_idx, ax in enumerate(axes):
        if ax_idx < len(wells):
            well = wells[ax_idx]
            x = df.loc[df[well_col] == well, value_col].to_numpy(dtype=float)

            ax.hist(x, bins=bin_edges)
            ax.set_title(f"{short_safe_id_label(well)} (n={len(x)})", fontsize=9)
            ax.set_xlim(bin_edges[0], bin_edges[-1])
            ax.set_ylim(0, y_max)
        else:
            ax.axis("off")

    fig.suptitle(title, fontsize=16)
    fig.supxlabel(xlabel)
    fig.supylabel("Number of units")

    fig.tight_layout(rect=[0, 0, 1, 0.96])

    save_or_show(
        filename=os.path.join(plot_dir, filename),
        show=show,
    )


def plot_snr_distribution(master, plot_dir, show=False, bins=50):
    _plot_distribution_per_well_grid(
        master=master,
        value_col="snr",
        plot_dir=plot_dir,
        filename="unit_snr_distribution_per_well.svg",
        xlabel="SNR",
        title="SNR distribution of QC-passing units per well",
        show=show,
        bins=bins,
        expected_n_wells=24,
    )


def plot_firing_rate_distribution(master, plot_dir, show=False, bins=50, title_extra=None):
    title = "Firing-rate distribution of QC-passing units per well"

    if title_extra:
        title += f"{title_extra}"

    _plot_distribution_per_well_grid(
        master=master,
        value_col="firing_rate",
        plot_dir=plot_dir,
        filename="unit_firing_rate_distribution_per_well.svg",
        xlabel="Firing rate (Hz)",
        title=title,
        show=show,
        bins=bins,
        expected_n_wells=24,
    )


In [58]:
### WELL-LEVEL PHENOTYPE PLOTS: median FR, mean FR, mean SNR, burst rate, mean network STTC
# ============================================================

def _plot_well_bar(
    well_summary,
    plot_dir,
    value_col,
    ylabel,
    title,
    filename,
    show=False,
    well_label_config=None,
):
    safe_col = pick_column(well_summary, ["safe_id", "well_id"])

    if safe_col is None or value_col is None:
        warnings.warn(f"Skipping {title}: required columns missing.")
        return

    if value_col not in well_summary.columns:
        warnings.warn(f"Skipping {title}: column '{value_col}' missing.")
        return

    df = well_summary.copy()
    df[safe_col] = df[safe_col].astype(str)
    df = df.sort_values(safe_col, key=lambda s: s.map(_well_sort_key))

    color_by_well, label_by_well = _build_well_label_maps(
        df=df,
        well_col=safe_col,
        well_label_config=well_label_config,
    )

    bar_colors = [
        color_by_well.get(well, None)
        for well in df[safe_col]
    ]

    plt.figure(figsize=(max(8, len(df) * 0.4), 4))

    if any(c is not None for c in bar_colors):
        plt.bar(df[safe_col], df[value_col], color=bar_colors)
    else:
        plt.bar(df[safe_col], df[value_col])

    plt.xlabel("Recording / well")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.xticks(rotation=90)

    if well_label_config is not None:
        legend_handles = []

        for label, spec in well_label_config.items():
            color = spec.get("color", None)
            if color is not None:
                legend_handles.append(
                    mpatches.Patch(color=color, label=label)
                )

        if len(legend_handles) > 0:
            plt.legend(
                handles=legend_handles,
                title="Group",
                bbox_to_anchor=(1.02, 1),
                loc="upper left",
                borderaxespad=0,
            )
            plt.tight_layout(rect=[0, 0, 0.85, 1])
        else:
            plt.tight_layout()
    else:
        plt.tight_layout()
    plt.rcParams['svg.fonttype'] = 'none'

    save_or_show(
        filename=os.path.join(plot_dir, filename),
        show=show,
    )


def plot_median_firing_rate_per_well(
    well_summary,
    plot_dir,
    show=False,
    well_label_config=None,
):
    fr_col = pick_column(
        well_summary,
        ["median_firing_rate_hz", "median_firing_rate"],
    )

    if fr_col is None:
        warnings.warn("Skipping median firing-rate per well: required columns missing.")
        return

    _plot_well_bar(
        well_summary=well_summary,
        plot_dir=plot_dir,
        value_col=fr_col,
        ylabel="Median firing rate (Hz)",
        title="Median firing rate per well",
        filename="well_median_firing_rate.svg",
        show=show,
        well_label_config=well_label_config,
    )


def plot_mean_firing_rate_per_well(
    well_summary,
    plot_dir,
    show=False,
    well_label_config=None,
):
    fr_col = pick_column(
        well_summary,
        ["mean_firing_rate_hz", "mean_firing_rate"],
    )

    if fr_col is None:
        warnings.warn("Skipping mean firing-rate per well: required columns missing.")
        return

    _plot_well_bar(
        well_summary=well_summary,
        plot_dir=plot_dir,
        value_col=fr_col,
        ylabel="Mean firing rate (Hz)",
        title="Mean firing rate per well",
        filename="well_mean_firing_rate.svg",
        show=show,
        well_label_config=well_label_config,
    )


def plot_mean_snr_per_well(
    well_summary,
    plot_dir,
    show=False,
    well_label_config=None,
):
    snr_col = pick_column(
        well_summary,
        ["mean_snr", "median_snr"],
    )

    if snr_col is None:
        warnings.warn("Skipping SNR per well: required columns missing.")
        return

    nice_titles = {
        "mean_snr": "Mean SNR",
        "median_snr": "Median SNR",
    }

    nice_title = nice_titles.get(
        snr_col,
        snr_col.replace("_", " ").title(),
    )

    _plot_well_bar(
        well_summary=well_summary,
        plot_dir=plot_dir,
        value_col=snr_col,
        ylabel=nice_title,
        title=f"{nice_title} per well",
        filename=f"well_{snr_col}.svg",
        show=show,
        well_label_config=well_label_config,
    )


def plot_burst_rate_per_well(
    well_summary,
    plot_dir,
    show=False,
    well_label_config=None,
):
    burst_col = pick_column(
        well_summary,
        ["mean_burst_rate_hz", "median_burst_rate_hz"],
    )

    if burst_col is None:
        warnings.warn("Skipping burst-rate per well: required columns missing.")
        return

    if burst_col == "mean_burst_rate_hz":
        nice_title = "Mean burst rate"
    elif burst_col == "median_burst_rate_hz":
        nice_title = "Median burst rate"
    else:
        nice_title = burst_col.replace("_", " ").title()

    _plot_well_bar(
        well_summary=well_summary,
        plot_dir=plot_dir,
        value_col=burst_col,
        ylabel=nice_title + '[Hz]',
        title=nice_title,
        filename=f"well_{burst_col}.svg",
        show=show,
        well_label_config=well_label_config,
    )

def plot_mean_network_sttc_per_well(
    well_summary,
    plot_dir,
    show=False,
    well_label_config=None,
):
    sttc_col = pick_column(
        well_summary,
        ["mean_network_sttc", "mean_unit_network_sttc", "mean_pairwise_sttc"],
    )

    if sttc_col is None:
        warnings.warn("Skipping network STTC per well: required columns missing.")
        return

    nice_titles = {
        "mean_network_sttc": "Mean network STTC",
        "mean_unit_network_sttc": "Mean unit network STTC",
        "mean_pairwise_sttc": "Mean pairwise STTC",
    }

    nice_title = nice_titles.get(
        sttc_col,
        sttc_col.replace("_", " ").title(),
    )

    _plot_well_bar(
        well_summary=well_summary,
        plot_dir=plot_dir,
        value_col=sttc_col,
        ylabel=nice_title,
        title=f"{nice_title} per well",
        filename=f"well_{sttc_col}.svg",
        show=show,
        well_label_config=well_label_config,
    )

In [59]:
### BURST / ISI distribution PLOTS per unit (all): burst rate, burst duration, coefficient of variation of the inter-spike intervals, firing vs burst rate
# ============================================================

def plot_burst_rate_distribution(master, plot_dir, show=False, bins=50):
    if "burst_rate_hz" not in master.columns:
        warnings.warn("Skipping burst-rate distribution: column 'burst_rate_hz' not found.")
        return

    plt.figure()
    plt.hist(master["burst_rate_hz"].dropna(), bins=bins)
    plt.xlabel("Burst rate (Hz)")
    plt.ylabel("Number of units")
    plt.title("Burst-rate distribution")
    plt.rcParams['svg.fonttype'] = 'none'

    save_or_show(
        filename=os.path.join(plot_dir, "unit_burst_rate_distribution.svg"),
        show=show,
    )


def plot_burst_duration_distribution(master, plot_dir, show=False, bins=50):
    if "mean_burst_duration_s" not in master.columns:
        warnings.warn("Skipping burst-duration distribution: column 'mean_burst_duration_s' not found.")
        return

    plt.figure()
    plt.hist(master["mean_burst_duration_s"].dropna(), bins=bins)
    plt.xlabel("Mean burst duration (s)")
    plt.ylabel("Number of units")
    plt.title("Mean burst-duration distribution")
    plt.rcParams['svg.fonttype'] = 'none'

    save_or_show(
        filename=os.path.join(plot_dir, "unit_burst_duration_distribution.svg"),
        show=show,
    )


def plot_cv_isi_distribution(master, plot_dir, show=False, bins=50):
    #how irregular the timing between spikes is
    if "cv_isi" not in master.columns:
        warnings.warn("Skipping CV ISI distribution: column 'cv_isi' not found.")
        return

    plt.figure()
    plt.hist(master["cv_isi"].dropna(), bins=bins)
    plt.xlabel("CV ISI")
    plt.ylabel("Number of units")
    plt.title("CV ISI distribution")
    plt.rcParams['svg.fonttype'] = 'none'
    save_or_show(
        filename=os.path.join(plot_dir, "unit_cv_isi_distribution.svg"),
        show=show,
    )


def plot_firing_rate_vs_burst_rate(master, plot_dir, show=False):
    if "firing_rate" not in master.columns or "burst_rate_hz" not in master.columns:
        warnings.warn("Skipping firing-rate vs burst-rate plot: required columns missing.")
        return

    plt.figure()
    plt.scatter(master["firing_rate"], master["burst_rate_hz"], s=20, alpha=0.7)
    plt.xlabel("Firing rate (Hz)")
    plt.ylabel("Burst rate (Hz)")
    plt.title("Firing rate vs burst rate")
    plt.rcParams['svg.fonttype'] = 'none'
    save_or_show(
        filename=os.path.join(plot_dir, "unit_firing_rate_vs_burst_rate.svg"),
        show=show,
    )


In [60]:
### BURST / ISI PLOTS per unit (per well): 
# ============================================================

import re


def _get_well_column(master):
    """
    Prefer well_id if present, otherwise fall back to safe_id.
    """
    if "well_id" in master.columns:
        return "well_id"
    if "safe_id" in master.columns:
        return "safe_id"
    return None


def _well_sort_key(well):
    """
    Natural sorting for names like:
    well000, well001, ...
    rec0000_well000, rec0000_well001, ...
    """
    well = str(well)
    nums = re.findall(r"\d+", well)

    if len(nums) == 0:
        return (well,)

    return tuple(int(x) for x in nums)


def _plot_hist_distribution_per_well_grid(
    master,
    value_col,
    plot_dir,
    filename,
    xlabel,
    title,
    show=False,
    bins=50,
    font_size=12,
    expected_n_wells=24,
):
    well_col = _get_well_column(master)

    if well_col is None:
        warnings.warn(
            f"Skipping {value_col} per-well plot: neither 'well_id' nor 'safe_id' found."
        )
        return

    if value_col not in master.columns:
        warnings.warn(
            f"Skipping {value_col} per-well plot: column '{value_col}' not found."
        )
        return

    df = master.copy()
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    df = df.dropna(subset=[value_col, well_col])

    if df.empty:
        warnings.warn(
            f"Skipping {value_col} per-well plot: no finite values found."
        )
        return

    df[well_col] = df[well_col].astype(str)

    wells = sorted(df[well_col].unique(), key=_well_sort_key)

    if len(wells) != expected_n_wells:
        warnings.warn(
            f"Expected {expected_n_wells} wells, but found {len(wells)} "
            f"unique values in '{well_col}'. Plotting available wells."
        )

    wells = wells[:expected_n_wells]

    all_values = df[value_col].to_numpy(dtype=float)

    # Shared x-axis bins across all wells
    x_min = 0
    x_max = np.nanmax(all_values) + 5

    bin_edges = np.linspace(x_min, x_max, bins + 1)

    # Shared y-axis limit across all wells
    max_count = 0
    for well in wells:
        x = df.loc[df[well_col] == well, value_col].to_numpy(dtype=float)
        counts, _ = np.histogram(x, bins=bin_edges)
        if len(counts) > 0:
            max_count = max(max_count, counts.max())

    y_max = max_count * 1.10 if max_count > 0 else 1

    fig, axes = plt.subplots(
        nrows=4,
        ncols=6,
        figsize=(20, 12),
        sharex=True,
        sharey=True,
    )

    axes = axes.ravel()

    for ax_idx, ax in enumerate(axes):
        if ax_idx < len(wells):
            well = wells[ax_idx]
            x = df.loc[df[well_col] == well, value_col].to_numpy(dtype=float)

            ax.hist(x, bins=bin_edges)
            ax.set_title(
                f"{short_safe_id_label(well)} (n={len(x)})",
                fontsize=font_size,
            )
            ax.tick_params(axis="both", labelsize=font_size)
            ax.set_xlim(x_min, x_max)
            ax.set_ylim(0, y_max)
        else:
            ax.axis("off")

    fig.suptitle(title, fontsize=font_size)
    fig.supxlabel(xlabel, fontsize=font_size)
    fig.supylabel("Number of units", fontsize=font_size)
    

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.rcParams['svg.fonttype'] = 'none'
    save_or_show(
        filename=os.path.join(plot_dir, filename),
        show=show,
    )


def _plot_scatter_per_well_grid(
    master,
    x_col,
    y_col,
    plot_dir,
    filename,
    xlabel,
    ylabel,
    title,
    show=False,
    expected_n_wells=24,
):
    well_col = _get_well_column(master)

    if well_col is None:
        warnings.warn(
            f"Skipping {x_col} vs {y_col} per-well plot: neither 'well_id' nor 'safe_id' found."
        )
        return

    missing = [col for col in [x_col, y_col] if col not in master.columns]
    if len(missing) > 0:
        warnings.warn(
            f"Skipping {x_col} vs {y_col} per-well plot: missing columns {missing}."
        )
        return

    df = master.copy()
    df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
    df[y_col] = pd.to_numeric(df[y_col], errors="coerce")
    df = df.dropna(subset=[x_col, y_col, well_col])

    if df.empty:
        warnings.warn(
            f"Skipping {x_col} vs {y_col} per-well plot: no finite values found."
        )
        return

    df[well_col] = df[well_col].astype(str)

    wells = sorted(df[well_col].unique(), key=_well_sort_key)

    if len(wells) != expected_n_wells:
        warnings.warn(
            f"Expected {expected_n_wells} wells, but found {len(wells)} "
            f"unique values in '{well_col}'. Plotting available wells."
        )

    wells = wells[:expected_n_wells]

    x_min = df[x_col].min()
    x_max = df[x_col].max()
    y_min = df[y_col].min()
    y_max = df[y_col].max()

    x_pad = 0.05 * (x_max - x_min) if x_max > x_min else 1
    y_pad = 0.05 * (y_max - y_min) if y_max > y_min else 1

    fig, axes = plt.subplots(
        nrows=4,
        ncols=6,
        figsize=(20, 12),
        sharex=True,
        sharey=True,
    )

    axes = axes.ravel()

    for ax_idx, ax in enumerate(axes):
        if ax_idx < len(wells):
            well = wells[ax_idx]
            sub = df.loc[df[well_col] == well]

            ax.scatter(
                sub[x_col],
                sub[y_col],
                s=20,
                alpha=0.7,
            )

            ax.set_title(f"{short_safe_id_label(well)} (n={len(sub)})", fontsize=9)
            ax.set_xlim(x_min - x_pad, x_max + x_pad)
            ax.set_ylim(y_min - y_pad, y_max + y_pad)
        else:
            ax.axis("off")

    fig.suptitle(title, fontsize=16)
    fig.supxlabel(xlabel)
    fig.supylabel(ylabel)

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.rcParams['svg.fonttype'] = 'none'
    save_or_show(
        filename=os.path.join(plot_dir, filename),
        show=show,
    )


def plot_burst_rate_distribution(master, plot_dir, show=False, bins=50):
    _plot_hist_distribution_per_well_grid(
        master=master,
        value_col="burst_rate_hz",
        plot_dir=plot_dir,
        filename="unit_burst_rate_distribution_per_well.svg",
        xlabel="Burst rate (Hz)",
        title="Burst-rate distribution per well",
        show=show,
        bins=bins,
        expected_n_wells=24,
    )


def plot_burst_duration_distribution(master, plot_dir, show=False, bins=50):
    _plot_hist_distribution_per_well_grid(
        master=master,
        value_col="mean_burst_duration_s",
        plot_dir=plot_dir,
        filename="unit_burst_duration_distribution_per_well.svg",
        xlabel="Mean burst duration (s)",
        title="Mean burst-duration distribution per well",
        show=show,
        bins=bins,
        expected_n_wells=24,
    )


def plot_cv_isi_distribution(master, plot_dir, show=False, bins=50):
    _plot_hist_distribution_per_well_grid(
        master=master,
        value_col="cv_isi",
        plot_dir=plot_dir,
        filename="unit_cv_isi_distribution_per_well.svg",
        xlabel="CV ISI",
        title="CV ISI distribution per well",
        show=show,
        bins=bins,
        expected_n_wells=24,
    )


def plot_firing_rate_vs_burst_rate(master, plot_dir, show=False):
    _plot_scatter_per_well_grid(
        master=master,
        x_col="firing_rate",
        y_col="burst_rate_hz",
        plot_dir=plot_dir,
        filename="unit_firing_rate_vs_burst_rate_per_well.svg",
        xlabel="Firing rate (Hz)",
        ylabel="Burst rate (Hz)",
        title="Firing rate vs burst rate per well",
        show=show,
        expected_n_wells=24,
    )

In [61]:
### SPATIAL PLOTS
# ============================================================

def plot_spatial_unit_map(master, plot_dir, safe_id=None, show=False):
    required = ["pos_x_um", "pos_y_um"]

    if not all(c in master.columns for c in required):
        warnings.warn("Skipping spatial unit map: position columns missing.")
        return

    if "safe_id" not in master.columns:
        warnings.warn("Skipping spatial unit map: column 'safe_id' missing.")
        return

    if safe_id is not None:
        groups = [(safe_id, master[master["safe_id"] == safe_id])]
    else:
        groups = list(master.groupby("safe_id"))

    for sid, sub in groups:
        if sub.empty:
            continue

        plt.figure()
        plt.scatter(sub["pos_x_um"], sub["pos_y_um"], s=20)
        plt.xlabel("x position [µm]")
        plt.ylabel("y position [µm]")
        plt.title(f"Spatial distribution of QC-passing units: {short_safe_id_label(sid)}")
        plt.gca().set_aspect("equal", adjustable="box")
        plt.rcParams['svg.fonttype'] = 'none'
        save_or_show(
            filename=os.path.join(plot_dir, f"{safe_filename(sid)}_spatial_unit_map.zvg"),
            show=show,
        )


def plot_spatial_firing_rate_map(master, plot_dir, safe_id=None, show=False):
    required = ["pos_x_um", "pos_y_um", "firing_rate"]

    if not all(c in master.columns for c in required):
        warnings.warn("Skipping spatial firing-rate map: required columns missing.")
        return

    if "safe_id" not in master.columns:
        warnings.warn("Skipping spatial firing-rate map: column 'safe_id' missing.")
        return

    if safe_id is not None:
        groups = [(safe_id, master[master["safe_id"] == safe_id])]
    else:
        groups = list(master.groupby("safe_id"))

    for sid, sub in groups:
        if sub.empty:
            continue

        plt.figure()
        sc = plt.scatter(
            sub["pos_x_um"],
            sub["pos_y_um"],
            c=sub["firing_rate"],
            s=25,
        )
        plt.colorbar(sc, label="Firing rate (Hz)")
        plt.xlabel("x position [µm]")
        plt.ylabel("y position [µm]")
        plt.title(f"Spatial firing-rate map: {short_safe_id_label(sid)}")
        plt.gca().set_aspect("equal", adjustable="box")
        plt.rcParams['svg.fonttype'] = 'none'

        save_or_show(
            filename=os.path.join(plot_dir, f"{safe_filename(sid)}_spatial_firing_rate_map.svg"),
            show=show,
        )


def plot_spatial_snr_map(master, plot_dir, safe_id=None, show=False):
    required = ["pos_x_um", "pos_y_um", "snr"]

    if not all(c in master.columns for c in required):
        warnings.warn("Skipping spatial SNR map: required columns missing.")
        return

    if "safe_id" not in master.columns:
        warnings.warn("Skipping spatial SNR map: column 'safe_id' missing.")
        return

    if safe_id is not None:
        groups = [(safe_id, master[master["safe_id"] == safe_id])]
    else:
        groups = list(master.groupby("safe_id"))

    for sid, sub in groups:
        if sub.empty:
            continue

        plt.figure()
        sc = plt.scatter(
            sub["pos_x_um"],
            sub["pos_y_um"],
            c=sub["snr"],
            s=25,
        )
        plt.colorbar(sc, label="SNR")
        plt.xlabel("x position [µm]")
        plt.ylabel("y position [µm]")
        plt.title(f"Spatial SNR map: {short_safe_id_label(sid)}")
        plt.gca().set_aspect("equal", adjustable="box")
        plt.rcParams['svg.fonttype'] = 'none'

        save_or_show(
            filename=os.path.join(plot_dir, f"{safe_filename(sid)}_spatial_snr_map.svg"),
            show=show,
        )
        
def plot_spatial_combined_map(
    master,
    plot_dir,
    safe_id=None,
    show=False,
):
    """
    Combined spatial plot per well:
        1. Unit locations
        2. Firing-rate map
        3. SNR map

    If safe_id=None, loops over all wells/safe_ids and saves one SVG per well.
    """

    from mpl_toolkits.axes_grid1.inset_locator import inset_axes

    safe_col = pick_column(master, ["safe_id", "well_id"])
    x_col = pick_column(master, ["pos_x_um", "x", "x_um"])
    y_col = pick_column(master, ["pos_y_um", "y", "y_um"])

    if safe_col is None or x_col is None or y_col is None:
        warnings.warn(
            "Skipping combined spatial plots: required columns missing "
            "(safe_id/well_id and x/y location columns)."
        )
        return

    if safe_id is None:
        safe_ids_plot = sorted(
            master[safe_col].dropna().astype(str).unique(),
            key=_well_sort_key,
        )
    else:
        safe_ids_plot = [str(safe_id)]

    for sid in safe_ids_plot:
        df = master.loc[master[safe_col].astype(str) == sid].copy()

        df[x_col] = pd.to_numeric(df[x_col], errors="coerce")
        df[y_col] = pd.to_numeric(df[y_col], errors="coerce")
        df = df.dropna(subset=[x_col, y_col])

        if df.empty:
            warnings.warn(f"Skipping combined spatial plot for {sid}: no valid positions.")
            continue

        has_fr = "firing_rate" in df.columns
        has_snr = "snr" in df.columns

        if has_fr:
            df["firing_rate"] = pd.to_numeric(df["firing_rate"], errors="coerce")

        if has_snr:
            df["snr"] = pd.to_numeric(df["snr"], errors="coerce")

        short_sid = short_safe_id_label(sid)

        fig, axes = plt.subplots(
            nrows=1,
            ncols=3,
            figsize=(18, 5.8),
            sharex=True,
            sharey=True,
            constrained_layout=False,
        )

        ax_loc, ax_fr, ax_snr = axes

        # Same x/y limits for all panels
        x_min, x_max = df[x_col].min(), df[x_col].max()
        y_min, y_max = df[y_col].min(), df[y_col].max()

        x_pad = 0.05 * max(x_max - x_min, 1)
        y_pad = 0.05 * max(y_max - y_min, 1)

        xlim = (x_min - x_pad, x_max + x_pad)
        ylim = (y_min - y_pad, y_max + y_pad)

        # ----------------------------------------------------
        # 1. Unit locations
        # ----------------------------------------------------
        ax_loc.scatter(
            df[x_col],
            df[y_col],
            s=26,
            alpha=0.8,
        )

        ax_loc.set_title("Unit locations")
        ax_loc.set_xlabel("x [µm]")
        ax_loc.set_ylabel("y [µm]")

        # ----------------------------------------------------
        # 2. Firing-rate map
        # ----------------------------------------------------
        if has_fr and df["firing_rate"].notna().any():
            sc_fr = ax_fr.scatter(
                df[x_col],
                df[y_col],
                c=df["firing_rate"],
                s=28,
                alpha=0.85,
            )

            cax_fr = inset_axes(
                ax_fr,
                width="34%",
                height="4%",
                loc="upper right",
                bbox_to_anchor=(0, 0.11, 1, 1),   # move colorbar slightly higher
                bbox_transform=ax_fr.transAxes,
                borderpad=0.8,
            )

            cbar_fr = fig.colorbar(
                sc_fr,
                cax=cax_fr,
                orientation="horizontal",
            )

            cbar_fr.set_label("Firing rate [Hz]", fontsize= 11, labelpad=3)
            cbar_fr.ax.tick_params(labelsize=11 , pad=1)
            cbar_fr.ax.xaxis.set_ticks_position("top")
            cbar_fr.ax.xaxis.set_label_position("top")

        else:
            ax_fr.scatter(
                df[x_col],
                df[y_col],
                s=26,
                alpha=0.8,
            )
            ax_fr.text(
                0.5,
                0.5,
                "No firing_rate",
                transform=ax_fr.transAxes,
                ha="center",
                va="center",
            )

        ax_fr.set_title("Firing rate")
        ax_fr.set_xlabel("x [µm]")

        # ----------------------------------------------------
        # 3. SNR map
        # ----------------------------------------------------
        if has_snr and df["snr"].notna().any():
            sc_snr = ax_snr.scatter(
                df[x_col],
                df[y_col],
                c=df["snr"],
                s=28,
                alpha=0.85,
            )

            cax_snr = inset_axes(
                ax_snr,
                width="34%",
                height="4%",
                loc="upper right",
                bbox_to_anchor=(0, 0.11, 1, 1),   # move colorbar slightly higher
                bbox_transform=ax_snr.transAxes,
                borderpad=0.8,
            )

            cbar_snr = fig.colorbar(
                sc_snr,
                cax=cax_snr,
                orientation="horizontal",
            )

            cbar_snr.set_label("SNR", fontsize=11, labelpad=3)
            cbar_snr.ax.tick_params(labelsize=11, pad=1)
            cbar_snr.ax.xaxis.set_ticks_position("top")
            cbar_snr.ax.xaxis.set_label_position("top")
            
        else:
            ax_snr.scatter(
                df[x_col],
                df[y_col],
                s=26,
                alpha=0.8,
            )
            ax_snr.text(
                0.5,
                0.5,
                "No snr",
                transform=ax_snr.transAxes,
                ha="center",
                va="center",
            )

        ax_snr.set_title("SNR")
        ax_snr.set_xlabel("x [µm]")

        # ----------------------------------------------------
        # Format all panels equally
        # ----------------------------------------------------
        for ax in axes:
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)
            ax.set_aspect("equal", adjustable="box")
            ax.grid(False)

        plt.setp(ax_fr.get_yticklabels(), visible=False)
        plt.setp(ax_snr.get_yticklabels(), visible=False)

        fig.suptitle(f"Spatial unit maps: {short_sid}", fontsize=15)

        fig.subplots_adjust(
            left=0.05,
            right=0.98,
            bottom=0.14,
            top=0.86,
            wspace=0.22,
        )

        save_or_show(
            filename=os.path.join(
                plot_dir,
                f"{safe_filename(sid)}_spatial_combined.svg",
            ),
            show=show,
        )

In [62]:
### GRAPH NETWORKS: HOW ARE THE UNITS CONNECTED

def plot_graph_metrics_summary_panel(
    master,
    plot_dir,
    filename="graph_metrics_summary_panel.svg",
    show=False,
    expected_n_wells=24,
    well_label_config=None,
):
    well_col = _get_well_column(master)

    required_cols = [
        "graph_degree",
        "graph_average_clustering",
        "graph_largest_component_fraction",
    ]

    if well_col is None:
        warnings.warn("Skipping graph metrics summary plot: no well column found.")
        return

    missing = [c for c in required_cols if c not in master.columns]
    if missing:
        warnings.warn(f"Skipping graph metrics summary plot. Missing columns: {missing}")
        return

    df = master.copy()
    df[well_col] = df[well_col].astype(str)

    for col in required_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=[well_col, "graph_degree"])

    if df.empty:
        warnings.warn("Skipping graph metrics summary plot: no finite graph degree values.")
        return

    wells = sorted(df[well_col].dropna().unique(), key=_well_sort_key)

    if len(wells) != expected_n_wells:
        warnings.warn(
            f"Expected {expected_n_wells} wells, but found {len(wells)}. "
            "Plotting available wells."
        )

    wells = wells[:expected_n_wells]

    color_by_well, label_by_well = _build_well_label_maps(
        df=pd.DataFrame({well_col: wells}),
        well_col=well_col,
        well_label_config=well_label_config,
    )

    bar_colors = [color_by_well.get(well, None) for well in wells]
    use_colors = any(c is not None for c in bar_colors)

    # ---------- Panel A: graph degree distribution heatmap ----------
    max_degree = int(np.nanmax(df["graph_degree"]))
    degree_bins = np.arange(0, max_degree + 1)

    heatmap = np.zeros((len(wells), len(degree_bins)))

    for i, well in enumerate(wells):
        x = df.loc[df[well_col] == well, "graph_degree"].dropna().astype(int)
        counts = x.value_counts()

        for j, degree in enumerate(degree_bins):
            heatmap[i, j] = counts.get(degree, 0)

    # ---------- Panels B/C: one scalar per well ----------
    summary = (
        df.groupby(well_col, as_index=False)
        .agg(
            graph_average_clustering=("graph_average_clustering", "first"),
            graph_largest_component_fraction=("graph_largest_component_fraction", "first"),
        )
    )

    summary[well_col] = pd.Categorical(
        summary[well_col],
        categories=wells,
        ordered=True,
    )
    summary = summary.sort_values(well_col)

    labels = [short_safe_id_label(well) for well in wells]
    x_pos = np.arange(len(wells))
    font_size = 12

    plt.rcParams.update({
        "font.size": font_size,
        "axes.titlesize": font_size,
        "axes.labelsize": font_size,
        "xtick.labelsize": font_size,
        "ytick.labelsize": font_size,
        "legend.fontsize": font_size,
        "legend.title_fontsize": font_size,
        "figure.titlesize": font_size,
    })

    fig = plt.figure(figsize=(18, 11))

    gs = fig.add_gridspec(
        nrows=2,
        ncols=3,
        height_ratios=[1.15, 1],
        width_ratios=[1, 1, 0.18],
        wspace=0.35,
        hspace=0.28,
    )

    ax_a = fig.add_subplot(gs[0, 0:2])
    ax_cbar = fig.add_subplot(gs[0, 2])

    ax_b = fig.add_subplot(gs[1, 0])
    ax_c = fig.add_subplot(gs[1, 1])
    ax_legend = fig.add_subplot(gs[1, 2])
    ax_legend.axis("off")

    # ---------- A ----------
    im = ax_a.imshow(
        heatmap,
        aspect="auto",
        interpolation="nearest",
        cmap="viridis",
    )

    ax_a.set_title("Distribution of inter-unit connections")
    ax_a.set_xlabel("Number of significant functional connections per unit")
    ax_a.set_ylabel("Recording / well")
    ax_a.set_xticks(np.arange(len(degree_bins)))
    ax_a.set_xticklabels(degree_bins)
    ax_a.set_yticks(np.arange(len(wells)))
    ax_a.set_yticklabels(labels)

    ax_cbar.axis("off")

    cbar = fig.colorbar(
        im,
        ax=ax_a,
        fraction=0.025,
        pad=0.02,
    )
    cbar.set_label("Number of units")
    # ---------- B ----------
    ax_b.bar(
        x_pos,
        summary["graph_average_clustering"].to_numpy(dtype=float),
        color=bar_colors if use_colors else None,
    )

    ax_b.set_title("Local network organization\n (0 > random connections; 1 > every nieghbour connects to every other nerigbour )")
    ax_b.set_ylabel("Average clustering coefficient")
    ax_b.set_ylim(0, 1)
    ax_b.set_xticks(x_pos)
    ax_b.set_xticklabels(labels, rotation=90)
    ax_b.set_xlabel("")

    # ---------- C ----------
    ax_c.bar(
        x_pos,
        summary["graph_largest_component_fraction"].to_numpy(dtype=float),
        color=bar_colors if use_colors else None,
    )

    ax_c.set_title("Fraction of units in main network")
    ax_c.set_ylabel("Largest connected component fraction")
    ax_c.set_ylim(0, 1)
    ax_c.set_xticks(x_pos)
    ax_c.set_xticklabels(labels, rotation=90)
    ax_c.set_xlabel("")

    # ---------- Legend ----------
    if well_label_config is not None:
        legend_handles = []

        for label, spec in well_label_config.items():
            color = spec.get("color", None)
            if color is not None:
                legend_handles.append(
                    mpatches.Patch(color=color, label=label)
                )

        if len(legend_handles) > 0:
            ax_legend.legend(
                handles=legend_handles,
                title="",
                loc="center left",
                bbox_to_anchor=(-0.65, 0.5),
                frameon=False,
            )

    fig.suptitle("Network connectivity summary", y=0.95)

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.rcParams['svg.fonttype'] = 'none'
    save_or_show(
        filename=os.path.join(plot_dir, filename),
        show=show,
    )

def plot_spatial_graph_metric_per_well_grid(
    master,
    plot_dir,
    value_col="graph_degree",
    filename=None,
    title=None,
    show=False,
    expected_n_wells=24,
    point_size=25,
    cmap="viridis",
    well_title_fontsize=12,
):
    well_col = _get_well_column(master)

    required_cols = [
        well_col,
        "pos_x_um",
        "pos_y_um",
        value_col,
    ]

    missing = [c for c in required_cols if c is None or c not in master.columns]
    if missing:
        warnings.warn(
            f"Skipping spatial graph plot for {value_col}: missing columns {missing}"
        )
        return

    df = master.copy()
    df[well_col] = df[well_col].astype(str)

    for col in ["pos_x_um", "pos_y_um", value_col]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=[well_col, "pos_x_um", "pos_y_um", value_col])

    if df.empty:
        warnings.warn(
            f"Skipping spatial graph plot for {value_col}: no finite values found."
        )
        return

    wells = sorted(df[well_col].unique(), key=_well_sort_key)

    if len(wells) != expected_n_wells:
        warnings.warn(
            f"Expected {expected_n_wells} wells, but found {len(wells)}. "
            "Plotting available wells."
        )

    wells = wells[:expected_n_wells]

    vmin = df[value_col].min()
    vmax = df[value_col].max()

    x_min, x_max = df["pos_x_um"].min(), df["pos_x_um"].max()
    y_min, y_max = df["pos_y_um"].min(), df["pos_y_um"].max()

    if filename is None:
        filename = f"spatial_{value_col}_per_well_grid.svg"

    if title is None:
        nice_title = value_col.replace("_", " ")
        title = f"Spatial distribution of {nice_title}"

    fig, axes = plt.subplots(
        nrows=4,
        ncols=6,
        figsize=(20, 12),
        sharex=True,
        sharey=True,
    )

    axes = axes.ravel()
    scatter_for_cbar = None

    for ax_idx, ax in enumerate(axes):
        if ax_idx < len(wells):
            well = wells[ax_idx]
            sub = df.loc[df[well_col] == well]

            scatter_for_cbar = ax.scatter(
                sub["pos_x_um"],
                sub["pos_y_um"],
                c=sub[value_col],
                s=point_size,
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                edgecolors="none",
            )

            ax.set_title(
                short_safe_id_label(well),
                fontsize=well_title_fontsize,
                
            )

            ax.set_xlim(x_min, x_max)
            ax.set_ylim(y_min, y_max)
            ax.set_aspect("equal", adjustable="box")
            ax.invert_yaxis()
        else:
            ax.axis("off")

    fig.suptitle(title, fontsize=14)
    fig.supxlabel("x position [µm]", fontsize = 12)
    fig.supylabel("y position [µm]", fontsize = 12)

    if scatter_for_cbar is not None:
        # [left, bottom, width, height] in figure coordinates
        cbar_ax = fig.add_axes([0.8, 0.96, 0.18, 0.018])

        cbar = fig.colorbar(
            scatter_for_cbar,
            cax=cbar_ax,
            orientation="horizontal",
        )

        cbar.set_label(value_col.replace("_", " "), fontsize=12)
        cbar.ax.tick_params(labelsize=12)

    fig.tight_layout(rect=[0, 0.05, 1, 0.91])
    plt.rcParams['svg.fonttype'] = 'none'

    save_or_show(
        filename=os.path.join(plot_dir, filename),
        show=show,
    )


In [63]:
### NETWORK SUMMARY ANALYSIS
# ============================================================

def summarize_sttc_networks(
    safe_ids,
    output_dir,
    table_dir,
    thresholds=(0.05, 0.10, 0.20),
):
    try:
        import networkx as nx
    except ImportError:
        warnings.warn("Skipping network summary: install networkx first.")
        return pd.DataFrame()

    rows = []

    for safe_id in safe_ids:
        try:
            sttc = load_sttc_matrix(safe_id, output_dir=output_dir)
        except FileNotFoundError:
            continue

        mat = sttc.values.astype(float)
        n = mat.shape[0]

        row = {
            "safe_id": safe_id,
            "n_units_network": n,
        }

        if n < 2:
            rows.append(row)
            continue

        upper = mat[np.triu_indices_from(mat, k=1)]
        upper = upper[np.isfinite(upper)]

        row["mean_pairwise_sttc"] = np.nanmean(upper)
        row["median_pairwise_sttc"] = np.nanmedian(upper)
        row["max_pairwise_sttc"] = np.nanmax(upper)
        row["frac_positive_sttc"] = np.mean(upper > 0)
        row["frac_negative_sttc"] = np.mean(upper < 0)

        for thr in thresholds:
            adj = mat > thr
            np.fill_diagonal(adj, False)

            G = nx.from_numpy_array(adj)
            label = str(thr).replace(".", "p")

            if G.number_of_nodes() > 0:
                comps = list(nx.connected_components(G))
                largest_component_fraction = max(len(c) for c in comps) / G.number_of_nodes()
            else:
                largest_component_fraction = np.nan

            row[f"edge_density_gt_{label}"] = nx.density(G)
            row[f"mean_degree_gt_{label}"] = np.mean([d for _, d in G.degree()]) if G.number_of_nodes() else np.nan
            row[f"mean_clustering_gt_{label}"] = nx.average_clustering(G) if G.number_of_edges() > 0 else 0.0
            row[f"largest_component_fraction_gt_{label}"] = largest_component_fraction

        rows.append(row)

    network_summary = pd.DataFrame(rows)

    network_summary.to_csv(
        os.path.join(table_dir, "sttc_network_summary.csv"),
        index=False
    )

    return network_summary

In [64]:
### PCA and outlier ANALYSES
# ============================================================

def plot_unit_level_pca(master, plot_dir, table_dir, show=False):
    try:
        from sklearn.preprocessing import StandardScaler
        from sklearn.decomposition import PCA
    except ImportError:
        warnings.warn("Skipping unit-level PCA: install scikit-learn first.")
        return

    candidate_features = [
        "snr",
        "firing_rate",
        "presence_ratio",
        "isi_violation",
        "cv_isi",
        "burst_rate_hz",
        "mean_burst_duration_s",
        "mean_short_isi_s",
        "mean_network_sttc",
    ]

    feature_cols = [c for c in candidate_features if c in master.columns]

    if len(feature_cols) < 2:
        warnings.warn("Skipping unit-level PCA: fewer than 2 available features.")
        return

    X = master[feature_cols].replace([np.inf, -np.inf], np.nan).dropna()

    if len(X) < 3:
        warnings.warn("Skipping unit-level PCA: fewer than 3 complete units.")
        return

    X_scaled = StandardScaler().fit_transform(X)

    pca = PCA(n_components=2)
    pcs = pca.fit_transform(X_scaled)

    plt.figure()
    plt.scatter(pcs[:, 0], pcs[:, 1], s=20, alpha=0.7)
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
    plt.title("Unit-level phenotype PCA")
    plt.rcParams['svg.fonttype'] = 'none'
    save_or_show(
        filename=os.path.join(plot_dir, "unit_level_pca.svg"),
        show=show,
    )

    pca_df = pd.DataFrame({
        "PC1": pcs[:, 0],
        "PC2": pcs[:, 1],
    }, index=X.index)

    meta_cols = [c for c in ["safe_id", "unit_id", "recording_id", "well_id", "condition", "celltype"] if c in master.columns]

    pca_out = pd.concat(
        [
            master.loc[X.index, meta_cols].reset_index(drop=True),
            pca_df.reset_index(drop=True)
        ],
        axis=1
    )

    pca_out.to_csv(
        os.path.join(table_dir, "unit_level_pca_scores.csv"),
        index=False
    )

    loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_cols,
        columns=["PC1_loading", "PC2_loading"],
    )

    loadings.to_csv(
        os.path.join(table_dir, "unit_level_pca_loadings.csv")
    )


def plot_unit_level_pca_per_well(master, plot_dir, table_dir, show=False, expected_n_wells=24):
    try:
        from sklearn.preprocessing import StandardScaler
        from sklearn.decomposition import PCA
    except ImportError:
        warnings.warn("Skipping unit-level PCA: install scikit-learn first.")
        return

    import re

    def _get_well_column(df):
        if "well_id" in df.columns:
            return "well_id"
        if "safe_id" in df.columns:
            return "safe_id"
        return None

    def _well_sort_key(well):
        well = str(well)
        nums = re.findall(r"\d+", well)

        if len(nums) == 0:
            return (well,)

        return tuple(int(x) for x in nums)

    candidate_features = [
        "snr",
        "firing_rate",
        "presence_ratio",
        "isi_violation",
        "cv_isi",
        "burst_rate_hz",
        "mean_burst_duration_s",
        "mean_short_isi_s",
        "mean_network_sttc",
    ]

    feature_cols = [c for c in candidate_features if c in master.columns]

    if len(feature_cols) < 2:
        warnings.warn("Skipping unit-level PCA: fewer than 2 available features.")
        return

    well_col = _get_well_column(master)

    if well_col is None:
        warnings.warn("Skipping per-well PCA plot: neither 'well_id' nor 'safe_id' found.")
        return

    # Keep only complete rows for the PCA features
    X = master[feature_cols].replace([np.inf, -np.inf], np.nan).dropna()

    if len(X) < 3:
        warnings.warn("Skipping unit-level PCA: fewer than 3 complete units.")
        return

    # Global PCA across all complete units
    X_scaled = StandardScaler().fit_transform(X)

    pca = PCA(n_components=2)
    pcs = pca.fit_transform(X_scaled)

    pca_df = pd.DataFrame(
        {
            "PC1": pcs[:, 0],
            "PC2": pcs[:, 1],
        },
        index=X.index,
    )

    # Metadata + PCA scores for output and plotting
    meta_cols = [
        c for c in [
            "safe_id",
            "unit_id",
            "recording_id",
            "well_id",
            "condition",
            "celltype",
        ]
        if c in master.columns
    ]

    pca_out = pd.concat(
        [
            master.loc[X.index, meta_cols].reset_index(drop=True),
            pca_df.reset_index(drop=True),
        ],
        axis=1,
    )

    pca_out.to_csv(
        os.path.join(table_dir, "unit_level_pca_scores.csv"),
        index=False,
    )

    loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_cols,
        columns=["PC1_loading", "PC2_loading"],
    )

    loadings.to_csv(
        os.path.join(table_dir, "unit_level_pca_loadings.csv")
    )

    # Plot PCA scores separated by well
    plot_df = pd.concat(
        [
            master.loc[X.index, [well_col]].copy(),
            pca_df,
        ],
        axis=1,
    )

    plot_df[well_col] = plot_df[well_col].astype(str)

    wells = sorted(plot_df[well_col].unique(), key=_well_sort_key)

    if len(wells) != expected_n_wells:
        warnings.warn(
            f"Expected {expected_n_wells} wells, but found {len(wells)} "
            f"unique values in '{well_col}'. Plotting available wells."
        )

    wells = wells[:expected_n_wells]

    pc1_min = plot_df["PC1"].min()
    pc1_max = plot_df["PC1"].max()
    pc2_min = plot_df["PC2"].min()
    pc2_max = plot_df["PC2"].max()

    pc1_pad = 0.05 * (pc1_max - pc1_min) if pc1_max > pc1_min else 1
    pc2_pad = 0.05 * (pc2_max - pc2_min) if pc2_max > pc2_min else 1

    fig, axes = plt.subplots(
        nrows=4,
        ncols=6,
        figsize=(20, 12),
        sharex=True,
        sharey=True,
    )

    axes = axes.ravel()

    for ax_idx, ax in enumerate(axes):
        if ax_idx < len(wells):
            well = wells[ax_idx]
            sub = plot_df.loc[plot_df[well_col] == well]

            ax.scatter(
                sub["PC1"],
                sub["PC2"],
                s=20,
                alpha=0.7,
            )

            ax.set_title(f"{short_safe_id_label(well)} (n={len(sub)})", fontsize=9)
            ax.set_xlim(pc1_min - pc1_pad, pc1_max + pc1_pad)
            ax.set_ylim(pc2_min - pc2_pad, pc2_max + pc2_pad)
        else:
            ax.axis("off")

    fig.suptitle("Unit-level phenotype PCA per well", fontsize=16)

    fig.supxlabel(
        f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)"
    )
    fig.supylabel(
        f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)"
    )

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    plt.rcParams['svg.fonttype'] = 'none'
    save_or_show(
        filename=os.path.join(plot_dir, "unit_level_pca_perwell.svg"),
        show=show,
    )
    
def plot_well_level_pca(
    well_summary,
    plot_dir,
    table_dir,
    show=False,
    well_label_config=None,
):
    try:
        from sklearn.preprocessing import StandardScaler
        from sklearn.decomposition import PCA
    except ImportError:
        warnings.warn("Skipping well-level PCA: install scikit-learn first.")
        return

    import matplotlib.patches as mpatches

    safe_col = pick_column(well_summary, ["safe_id", "well_id"])

    candidate_features = [
        "n_units",
        "mean_firing_rate_hz",
        "median_firing_rate_hz",
        "mean_snr",
        "median_snr",
        "mean_burst_rate_hz",
        "median_burst_rate_hz",
        "mean_network_sttc",
        "mean_unit_network_sttc",
        "mean_pairwise_sttc",
    ]

    feature_cols = [c for c in candidate_features if c in well_summary.columns]

    if safe_col is None:
        warnings.warn("Skipping well-level PCA: no safe_id or well_id column found.")
        return

    if len(feature_cols) < 2:
        warnings.warn("Skipping well-level PCA: fewer than 2 available features.")
        return

    df = well_summary.copy()
    df[safe_col] = df[safe_col].astype(str)

    X = df[feature_cols].replace([np.inf, -np.inf], np.nan).dropna()

    if len(X) < 3:
        warnings.warn("Skipping well-level PCA: fewer than 3 complete wells.")
        return

    X_scaled = StandardScaler().fit_transform(X)

    pca = PCA(n_components=2)
    pcs = pca.fit_transform(X_scaled)

    pca_df = pd.DataFrame(
        {
            safe_col: df.loc[X.index, safe_col].values,
            "PC1": pcs[:, 0],
            "PC2": pcs[:, 1],
        },
        index=X.index,
    )

    color_by_well, label_by_well = _build_well_label_maps(
        df=df.loc[X.index],
        well_col=safe_col,
        well_label_config=well_label_config,
    )

    pca_df["group"] = pca_df[safe_col].map(label_by_well).fillna("Unlabelled")
    pca_df["color"] = pca_df[safe_col].map(color_by_well)

    plt.figure(figsize=(9, 7))

    if well_label_config is not None:
        for label, spec in well_label_config.items():
            color = spec.get("color", None)
            sub = pca_df[pca_df["group"] == label]

            if len(sub) == 0:
                continue

            plt.scatter(
                sub["PC1"],
                sub["PC2"],
                s=70,
                alpha=0.85,
                color=color,
                label=label,
            )

        unlabelled = pca_df[pca_df["group"] == "Unlabelled"]
        if len(unlabelled) > 0:
            plt.scatter(
                unlabelled["PC1"],
                unlabelled["PC2"],
                s=70,
                alpha=0.85,
                label="Unlabelled",
            )

        plt.legend(
            title="Group",
            bbox_to_anchor=(1.02, 1),
            loc="upper left",
            borderaxespad=0,
        )

    else:
        plt.scatter(
            pca_df["PC1"],
            pca_df["PC2"],
            s=70,
            alpha=0.85,
        )

    # Label each PCA point with only the well ID, e.g. well003
    for _, row in pca_df.iterrows():
        well_label = short_safe_id_label(row[safe_col])

        m = re.search(r"(well\d+)", well_label)
        if m:
            well_label = m.group(1)

        plt.text(
            row["PC1"],
            row["PC2"],
            well_label,
            fontsize=8,
            ha="left",
            va="bottom",
        )

    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
    plt.title("Well-level phenotype PCA")

    plt.tight_layout(rect=[0, 0, 0.85, 1])
    plt.rcParams['svg.fonttype'] = 'none'
    save_or_show(
        filename=os.path.join(plot_dir, "well_level_pca.svg"),
        show=show,
    )

    pca_df.drop(columns=["color"], errors="ignore").to_csv(
        os.path.join(table_dir, "well_level_pca_scores.csv"),
        index=False,
    )

    loadings = pd.DataFrame(
        pca.components_.T,
        index=feature_cols,
        columns=["PC1_loading", "PC2_loading"],
    )

    loadings.to_csv(
        os.path.join(table_dir, "well_level_pca_loadings.csv")
    )

# ============================================================
# 9. OUTLIER DETECTION
# ============================================================

def detect_well_outliers(well_summary, table_dir, z_threshold=3.0):
    try:
        from scipy.stats import zscore
    except ImportError:
        warnings.warn("Skipping well outlier detection: install scipy first.")
        return pd.DataFrame()

    candidate_cols = [
        "n_units",
        "n_qc_units",
        "mean_snr",
        "median_snr",
        "mean_firing_rate_hz",
        "median_firing_rate_hz",
        "mean_firing_rate",
        "median_firing_rate",
        "mean_burst_rate_hz",
        "mean_network_sttc",
        "mean_pairwise_sttc",
    ]

    cols = [c for c in candidate_cols if c in well_summary.columns]

    if len(cols) == 0:
        warnings.warn("Skipping outlier detection: no suitable numeric columns.")
        return pd.DataFrame()

    df = well_summary.copy()

    z = df[cols].apply(
        lambda x: zscore(x, nan_policy="omit")
    )

    outlier_mask = (np.abs(z) > z_threshold).any(axis=1)

    meta_cols = [c for c in ["safe_id", "recording_id", "well_id", "condition", "celltype"] if c in df.columns]

    outliers = df.loc[outlier_mask, meta_cols + cols].copy()

    outliers.to_csv(
        os.path.join(table_dir, "well_outliers.csv"),
        index=False
    )

    z_scores = pd.concat(
        [
            df[meta_cols].reset_index(drop=True),
            z.add_prefix("z_").reset_index(drop=True)
        ],
        axis=1
    )

    z_scores.to_csv(
        os.path.join(table_dir, "well_feature_z_scores.csv"),
        index=False
    )

    return outliers


In [65]:
### RASTER / POPULATION RATE HELPERS
# ============================================================

def plot_raster_from_sorting(
    safe_id,
    output_dir,
    plot_dir,
    sampling_rate=10000.0,
    max_units=50,
    show=False,
):
    """
    Requires saved Kilosort output and SpikeInterface.
    """
    try:
        import spikeinterface.full as si
    except ImportError:
        warnings.warn("Skipping raster plot: SpikeInterface not available.")
        return

    ks4_folder = os.path.join(output_dir, f"ks4_{safe_id}")

    if not os.path.exists(ks4_folder):
        warnings.warn(f"Skipping raster plot: sorter folder not found: {ks4_folder}")
        return

    try:
        sorting = si.read_sorter_folder(ks4_folder)
    except Exception:
        sorter_output = os.path.join(ks4_folder, "sorter_output")
        sorting = si.read_kilosort(sorter_output)

    unit_ids = list(sorting.unit_ids)

    plt.figure(figsize=(10, 6))

    for i, unit_id in enumerate(unit_ids):
        spikes = sorting.get_unit_spike_train(unit_id) / sampling_rate
        plt.vlines(spikes, i + 0.5, i + 1.5)

    plt.xlabel("Time (s)")
    plt.ylabel("Unit")
    plt.title(f"Raster plot: {safe_id}")
    plt.rcParams['svg.fonttype'] = 'none'
    save_or_show(
        filename=os.path.join(plot_dir, f"{safe_filename(safe_id)}_raster.svg"),
        show=show,
    )


def plot_population_rate_from_sorting(
    safe_id,
    output_dir,
    plot_dir,
    table_dir,
    sampling_rate=10000.0,
    duration_s=300.0,
    bin_size_s=1.0,
    max_units=None,
    show=False,
):
    """
    Requires saved Kilosort output and SpikeInterface.
    """
    try:
        import spikeinterface.full as si
    except ImportError:
        warnings.warn("Skipping population-rate plot: SpikeInterface not available.")
        return

    ks4_folder = os.path.join(output_dir, f"ks4_{safe_id}")

    if not os.path.exists(ks4_folder):
        warnings.warn(f"Skipping population-rate plot: sorter folder not found: {ks4_folder}")
        return

    try:
        sorting = si.read_sorter_folder(ks4_folder)
    except Exception:
        sorter_output = os.path.join(ks4_folder, "sorter_output")
        sorting = si.read_kilosort(sorter_output)

    unit_ids = list(sorting.unit_ids)

    if max_units is not None:
        unit_ids = unit_ids[:max_units]

    all_spikes = []

    for unit_id in unit_ids:
        spikes = sorting.get_unit_spike_train(unit_id) / sampling_rate
        all_spikes.extend(spikes)

    bins = np.arange(0, duration_s + bin_size_s, bin_size_s)
    counts, edges = np.histogram(all_spikes, bins=bins)
    rate = counts / bin_size_s

    plt.figure()
    plt.plot(edges[:-1], rate)
    plt.xlabel("Time (s)")
    plt.ylabel("Population spikes/s")
    plt.title(f"Population activity: {safe_id}")
    plt.rcParams['svg.fonttype'] = 'none'
    save_or_show(
        filename=os.path.join(plot_dir, f"{safe_filename(safe_id)}_population_rate.svg"),
        show=show,
    )

    rate_df = pd.DataFrame({
        "time_s": edges[:-1],
        "population_spikes_per_s": rate,
    })

    rate_df.to_csv(
        os.path.join(table_dir, f"{safe_filename(safe_id)}_population_rate.csv"),
        index=False,
    )

def plot_raster_and_population_rate_from_sorting(
    safe_id,
    output_dir,
    plot_dir,
    table_dir,
    sampling_rate=10000.0,
    duration_s=300.0,
    bin_size_s=1.0,
    max_units_raster=50,
    max_units_rate=None,
    show=False,
):
    """
    Combined raster + population-rate plot for one well.

    Top: raster plot using vertical spike lines.
    Bottom: population activity in spikes/s.
    """
    try:
        import spikeinterface.full as si
    except ImportError:
        warnings.warn("Skipping raster/population plot: SpikeInterface not available.")
        return

    ks4_folder = os.path.join(output_dir, f"ks4_{safe_id}")

    if not os.path.exists(ks4_folder):
        warnings.warn(f"Skipping raster/population plot: sorter folder not found: {ks4_folder}")
        return

    try:
        sorting = si.read_sorter_folder(ks4_folder)
    except Exception:
        sorter_output = os.path.join(ks4_folder, "sorter_output")
        sorting = si.read_kilosort(sorter_output)

    all_unit_ids = list(sorting.unit_ids)

    if len(all_unit_ids) == 0:
        warnings.warn(f"Skipping raster/population plot: no units found for {safe_id}")
        return

    # Units for raster
    raster_unit_ids = all_unit_ids

    # Units for population rate
    rate_unit_ids = all_unit_ids
    

    # --------------------------------------------------------
    # Population rate
    # --------------------------------------------------------
    all_spikes = []

    for unit_id in rate_unit_ids:
        spikes = sorting.get_unit_spike_train(unit_id) / sampling_rate
        spikes = np.asarray(spikes, dtype=float)

        spikes = spikes[
            np.isfinite(spikes) &
            (spikes >= 0) &
            (spikes <= duration_s)
        ]

        all_spikes.extend(spikes)

    bins = np.arange(0, duration_s + bin_size_s, bin_size_s)
    counts, edges = np.histogram(all_spikes, bins=bins)
    rate = counts / bin_size_s

    rate_df = pd.DataFrame({
        "time_s": edges[:-1],
        "population_spikes_per_s": rate,
    })

    rate_df.to_csv(
        os.path.join(table_dir, f"{safe_filename(safe_id)}_population_rate.csv"),
        index=False,
    )

    # --------------------------------------------------------
    # Combined plot
    # --------------------------------------------------------
    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=(11, 7),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1]},
    )

    ax_raster = axes[0]
    ax_rate = axes[1]

    # Raster: use vertical lines, same style as your original function
    for i, unit_id in enumerate(raster_unit_ids):
        spikes = sorting.get_unit_spike_train(unit_id) / sampling_rate
        spikes = np.asarray(spikes, dtype=float)

        spikes = spikes[
            np.isfinite(spikes) &
            (spikes >= 0) &
            (spikes <= duration_s)
        ]

        ax_raster.vlines(
            spikes,
            i + 0.5,
            i + 1.5,
            linewidth=0.6,
        )

    ax_raster.set_ylabel("Unit")
    ax_raster.set_ylim(0.5, len(raster_unit_ids) + 1.5)
    ax_raster.set_xlim(0, duration_s)

    title_id = short_safe_id_label(safe_id) if "short_safe_id_label" in globals() else safe_id
    ax_raster.set_title(f"Raster and population activity: {title_id}")


    # Population rate
    ax_rate.plot(edges[:-1], rate, linewidth=1.2)
    ax_rate.set_xlabel("Time (s)")
    ax_rate.set_ylabel("Population\nspikes/s")
    ax_rate.set_xlim(0, duration_s)

    fig.tight_layout()
    plt.rcParams['svg.fonttype'] = 'none'
    save_or_show(
        filename=os.path.join(
            plot_dir,
            f"{safe_filename(safe_id)}_raster_population_rate.svg",
        ),
        show=show,
    )



In [66]:
# DEFAULT CONFIGURATION
# ============================================================

DEFAULT_RAW_H5_FILE = "../Data/FT_SN_FabryLinesH9/260527/T003708/Network/000039/data.raw.h5"
DEFAULT_MASTER_PATH = "phenotype_matrix.csv"
DEFAULT_OUTPUT_DIR = "maxtwo_pipeline_results_sttc50_units100_minspikes5"


In [67]:
#SETUP PATHS
network_threshold = 0.10


def make_plots_output_dir_from_raw(raw_h5_file, base_output_dir):
    """
    Example:
        ../Data/FT_SN_FabryLinesH9/260527/T003708/Network/000039/data.raw.h5

    becomes:
        maxtwo_pipeline_results/plots_FT_SN_FabryLinesH9_260527__T003708__Network__000039__data
    """
    p = Path(raw_h5_file)

    parent_folder = p.parts[-6]
    date_id = p.parts[-5]
    time_id = p.parts[-4]
    scan_type = p.parts[-3]
    scan_id = p.parts[-2]
    file_stem = p.name.replace(".raw.h5", "").replace(".h5", "")

    experiment_id = f"plots_{parent_folder}_{date_id}__{time_id}__{scan_type}__{scan_id}__{file_stem}"

    return os.path.join(base_output_dir, experiment_id)


def get_metrics_output_dir_from_raw(raw_h5_file, base_output_dir):
    """
    Example:
        ../Data/FT_SN_FabryLinesH9/260527/T003708/Network/000039/data.raw.h5

    becomes:
        maxtwo_pipeline_results/260527__T003708__Network__000039__data
    """
    p = Path(raw_h5_file)

    date_id = p.parts[-5]
    time_id = p.parts[-4]
    scan_type = p.parts[-3]
    scan_id = p.parts[-2]
    file_stem = p.name.replace(".raw.h5", "").replace(".h5", "")

    experiment_id = f"{date_id}__{time_id}__{scan_type}__{scan_id}__{file_stem}"

    return os.path.join(base_output_dir, experiment_id)


def get_dataset_label(raw_h5_file):
    """
    Example:
        ../Data/FT_SN_FabryLinesH9/260527/T003708/Network/000039/data.raw.h5

    becomes:
        FT_SN_FabryLinesH9_260527__T003708__Network__000039__data
    """
    p = Path(raw_h5_file)

    parent_folder = p.parts[-6]
    date_id = p.parts[-5]
    time_id = p.parts[-4]
    scan_type = p.parts[-3]
    scan_id = p.parts[-2]
    file_stem = p.name.replace(".raw.h5", "").replace(".h5", "")

    experiment_id = f"{parent_folder}_{date_id}__{time_id}__{scan_type}__{scan_id}__{file_stem}"

    return os.path.join( experiment_id)


plot_dir = make_plots_output_dir_from_raw(
    raw_h5_file=DEFAULT_RAW_H5_FILE,
    base_output_dir=DEFAULT_OUTPUT_DIR,
)
table_dir = plot_dir
os.makedirs(plot_dir, exist_ok=True)



master_dir = get_metrics_output_dir_from_raw(
    raw_h5_file=DEFAULT_RAW_H5_FILE,
    base_output_dir=DEFAULT_OUTPUT_DIR,
)

master_path = os.path.join(master_dir, "phenotype_matrix.csv")
well_summary_path = os.path.join(master_dir , "well_level_summary.csv")


dataset_label= get_dataset_label(raw_h5_file=DEFAULT_RAW_H5_FILE,)

print("Dataset label:", dataset_label)
print("Experiment output directory:", master_dir)
print("Plot output directory:", plot_dir)
print("Table output directory:", table_dir)
print("Master path:", master_path)
print("Well summary path:", well_summary_path)

Dataset label: FT_SN_FabryLinesH9_260527__T003708__Network__000039__data
Experiment output directory: maxtwo_pipeline_results_sttc50_units100_minspikes5/260527__T003708__Network__000039__data
Plot output directory: maxtwo_pipeline_results_sttc50_units100_minspikes5/plots_FT_SN_FabryLinesH9_260527__T003708__Network__000039__data
Table output directory: maxtwo_pipeline_results_sttc50_units100_minspikes5/plots_FT_SN_FabryLinesH9_260527__T003708__Network__000039__data
Master path: maxtwo_pipeline_results_sttc50_units100_minspikes5/260527__T003708__Network__000039__data/phenotype_matrix.csv
Well summary path: maxtwo_pipeline_results_sttc50_units100_minspikes5/260527__T003708__Network__000039__data/well_level_summary.csv


In [68]:
# WELL GROUP COLOURS / LEGEND
# ============================================================

well_label_config_1193_284 = {
    "CORR_1193": {
        "wells": [1, 2, 3, 4, 5],
        "color": "#56B4E9",
    },
    
    "PAT_1193": {
        "wells": [7, 8, 9, 10, 11],
        "color": "#E69F00",
    },
    "CORR_284": {
        "wells": [13, 14, 15, 16, 17],
        "color": "#0072B2",
    },
    "PAT_284": {
        "wells": [19, 20, 21, 22, 23],
        "color": "#D55E00",
    },
    "H9": {
        "wells": [6, 12, 18, 24],
        "color": "#CC79A7",
    },
}

In [69]:
# LOAD DATA
# ============================================================

master = load_master(
    master_path=master_path,
    output_dir=master_dir,
)

well_summary = load_well_summary(
    master=master,
    well_summary_path=well_summary_path,
)

safe_ids = get_safe_ids(master)

print("Loaded master:", master.shape)
print("Loaded well_summary:", well_summary.shape)
print("Number of safe_ids:", len(safe_ids))

well_summary has no safe_id column. Reconstructing safe_id...
Loaded master: (6532, 85)
Loaded well_summary: (24, 74)
Number of safe_ids: 24


In [70]:
# RUN ONLY PER-WELL PLOTS
# ============================================================
show = False

# QC plots — these should now be 4x6 per-well grids
plot_snr_distribution(master, plot_dir=plot_dir, show=show)
plot_firing_rate_distribution(master, plot_dir=plot_dir, show=show)


for safe_id in safe_ids:
    plot_raster_and_population_rate_from_sorting(
        safe_id=safe_id,
        output_dir=master_dir,
        plot_dir=plot_dir,
        table_dir=table_dir,
        sampling_rate=10000.0,
        duration_s=300.0,
        bin_size_s=1.0,
        max_units_raster=50,
        show=show,
    )

plot_units_per_well(
    master,
    plot_dir=plot_dir,
    table_dir=table_dir,
    show=show,
    well_label_config=well_label_config_1193_284,
)
# Well-level plots
plot_median_firing_rate_per_well(
    well_summary,
    plot_dir=plot_dir,
    show=show,
    well_label_config=well_label_config_1193_284,
)

plot_mean_firing_rate_per_well(
    well_summary,
    plot_dir=plot_dir,
    show=show,
    well_label_config=well_label_config_1193_284,
)

plot_mean_snr_per_well(
    well_summary,
    plot_dir=plot_dir,
    show=show,
    well_label_config=well_label_config_1193_284,
)

plot_burst_rate_per_well(
    well_summary,
    plot_dir=plot_dir,
    show=show,
    well_label_config=well_label_config_1193_284,
)

plot_mean_network_sttc_per_well(
    well_summary,
    plot_dir=plot_dir,
    show=show,
    well_label_config=well_label_config_1193_284,
)

# Burst / ISI plots — 4x6 per-well grids
plot_burst_rate_distribution(master, plot_dir=plot_dir, show=show)
plot_burst_duration_distribution(master, plot_dir=plot_dir, show=show)
plot_cv_isi_distribution(master, plot_dir=plot_dir, show=show)
plot_firing_rate_vs_burst_rate(master, plot_dir=plot_dir, show=show)

# Spatial plots per well

plot_spatial_combined_map(
    master,
    plot_dir=plot_dir,
    safe_id=None,
    show=show,
)


network_summary = summarize_sttc_networks(
    safe_ids=safe_ids,
    output_dir=master_dir,
    table_dir=table_dir,
    thresholds=(0.05, 0.10, 0.20),
)

if not network_summary.empty:
    merged = well_summary.merge(
        network_summary,
        on="safe_id",
        how="left",
    )

    merged.to_csv(
        os.path.join(table_dir, "well_summary_with_network_metrics.csv"),
        index=False,
    )

#unit graphs
plot_graph_metrics_summary_panel(
    master,
    plot_dir=plot_dir,
    show=show,
    well_label_config=well_label_config_1193_284,
)

plot_spatial_graph_metric_per_well_grid(
    master,
    plot_dir=plot_dir,
    value_col="graph_degree",
    title="Spatial distribution of inter-unit connectivity",
    show=show,
)
plot_spatial_graph_metric_per_well_grid(
    master,
    plot_dir=plot_dir,
    value_col="graph_weighted_sttc_strength",
    title="Spatial distribution of weighted functional connectivity",
    show=show,
)

# PCA — use the per-well PCA, not the global unit-level PCA
plot_unit_level_pca_per_well(
    master=master,
    plot_dir=plot_dir,
    table_dir=table_dir,
    show=show,
)

plot_well_level_pca(
    well_summary=well_summary,
    plot_dir=plot_dir,
    table_dir=table_dir,
    show=show,
    well_label_config=well_label_config_1193_284,
)

# Outliers
outliers = detect_well_outliers(
    well_summary=well_summary,
    table_dir=table_dir,
    z_threshold=3.0,
)

if not outliers.empty:
    print("Potential well outliers detected:")
    print(outliers)
else:
    print("No extreme well outliers detected with z-threshold 3.0.")

/tmp/ipykernel_251535/3261405399.py:20: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_251535/3261405399.py:20: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_251535/3261405399.py:20: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_251535/3261405399.py:20: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_251535/3261405399.py:20: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
/tmp/ipykernel_251535/3261405399.py:20: UserWarning: This figure includes Axes that are not compatible with tight_layout, so resul

Potential well outliers detected:
           safe_id recording_id  well_id       condition          celltype  \
3  rec0000_well003      rec0000  well003  same_condition  iPSC_DRG_neurons   

   n_units  mean_snr  median_snr  mean_firing_rate  median_firing_rate  \
3       56  6.897083    6.293958          0.496243            0.033329   

   mean_burst_rate_hz  mean_network_sttc  
3            0.066241           0.010221  
